# Notebook 03: Stage 1 - Fairness Under Drift

**Tests H1:** Does distribution drift widen fairness gaps between demographic subgroups?

- Computes EOD, DPD, ECE per subgroup per window
- Permutation ANOVA for interaction (window x subgroup)
- Decision rule: interaction p < 0.0056 AND EOD >= 0.05 for minority by W4


In [ ]:
import sys, os, pickle
if 'google.colab' in sys.modules:
    sys.path.insert(0, '/content/FairDrift-code')
else:
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src import config
from src.fairness_metrics import compute_fairness_over_windows
from src.temporal_windows import construct_windows

# Load checkpoints
with open('../outputs/checkpoint_01.pkl', 'rb') as f:
    ckpt01 = pickle.load(f)
with open('../outputs/checkpoint_02.pkl', 'rb') as f:
    ckpt02 = pickle.load(f)

windows = ckpt01['windows']
feature_cols = ckpt01['feature_cols']
models = ckpt02['models']
scaler = ckpt02['scaler']
print('Loaded checkpoints')


## 1. Compute Fairness Metrics Across All Windows

In [ ]:
from src.fairness_metrics import (
    equalized_odds_difference, demographic_parity_difference,
    expected_calibration_error, compute_all_fairness_metrics
)

all_fairness = []

for (model_type, task), model in models.items():
    for w_idx in range(1, len(windows)):  # W2-W5
        window = windows[w_idx]
        X = window[feature_cols].values
        y_true = window[task].values

        # Scale if needed
        if model_type in ['logistic_regression', 'mlp']:
            X_eval = scaler.transform(X)
        else:
            X_eval = X

        y_proba = model.predict_proba(X_eval)[:, 1]
        y_pred = (y_proba >= 0.5).astype(int)

        # By Race
        race = window['race'].values
        metrics = compute_all_fairness_metrics(
            y_true, y_pred, y_proba, race, config.REFERENCE_RACE
        )
        metrics['window'] = w_idx + 1
        metrics['model'] = model_type
        metrics['task'] = task
        metrics['protected_attr'] = 'race'
        all_fairness.append(metrics)

        # By Gender
        gender = window['gender'].values
        metrics_g = compute_all_fairness_metrics(
            y_true, y_pred, y_proba, gender, config.REFERENCE_GENDER
        )
        metrics_g['window'] = w_idx + 1
        metrics_g['model'] = model_type
        metrics_g['task'] = task
        metrics_g['protected_attr'] = 'gender'
        all_fairness.append(metrics_g)

fairness_df = pd.concat(all_fairness, ignore_index=True)
print(f'Computed {len(fairness_df)} fairness metric rows')
fairness_df.head(10)


## 2. EOD Trajectory Across Windows (Core H1 Evidence)

In [ ]:
# Plot EOD trajectory for African American across windows
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

aa_data = fairness_df[
    (fairness_df['group'] == 'AfricanAmerican') & 
    (fairness_df['protected_attr'] == 'race')
]

for idx, task in enumerate(config.TASKS):
    ax = axes[idx]
    task_data = aa_data[aa_data['task'] == task]
    
    for model_type in config.MODEL_TYPES:
        model_data = task_data[task_data['model'] == model_type]
        ax.plot(model_data['window'], model_data['eod'], 'o-', label=model_type)
    
    ax.axhline(y=config.EOD_VIOLATION_THRESHOLD, color='red', linestyle='--', 
               label=f'Violation threshold ({config.EOD_VIOLATION_THRESHOLD})')
    ax.set_title(config.TASKS[task]['description'])
    ax.set_xlabel('Window')
    ax.set_ylabel('EOD (vs Caucasian)')
    ax.legend(fontsize=8)
    ax.set_xticks([2,3,4,5])

plt.suptitle('Equalized Odds Difference: African American vs Caucasian', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/fig04_eod_trajectory.png', dpi=300, bbox_inches='tight')
plt.show()


## 3. Permutation ANOVA (H1 Formal Test)

In [ ]:
def permutation_anova_interaction(metric_values, window_labels, subgroup_labels, n_perm=10000, seed=42):
    """Two-way permutation ANOVA testing window x subgroup interaction."""
    rng = np.random.default_rng(seed)
    
    # Observed F-statistic for interaction
    from itertools import product
    groups = {}
    for w, s in product(np.unique(window_labels), np.unique(subgroup_labels)):
        mask = (window_labels == w) & (subgroup_labels == s)
        if mask.sum() > 0:
            groups[(w, s)] = metric_values[mask]
    
    # Grand mean and cell means
    grand_mean = metric_values.mean()
    
    # SS interaction (simplified: difference of cell means from additive model)
    unique_w = np.unique(window_labels)
    unique_s = np.unique(subgroup_labels)
    
    def compute_ss_interaction(vals, w_labs, s_labs):
        gm = vals.mean()
        w_means = {w: vals[w_labs == w].mean() for w in unique_w}
        s_means = {s: vals[s_labs == s].mean() for s in unique_s}
        ss = 0
        for w in unique_w:
            for s in unique_s:
                mask = (w_labs == w) & (s_labs == s)
                n_cell = mask.sum()
                if n_cell > 0:
                    cell_mean = vals[mask].mean()
                    expected = w_means[w] + s_means[s] - gm
                    ss += n_cell * (cell_mean - expected) ** 2
        return ss
    
    observed_ss = compute_ss_interaction(metric_values, window_labels, subgroup_labels)
    
    # Permutation test
    count_ge = 0
    for _ in range(n_perm):
        perm_vals = rng.permutation(metric_values)
        perm_ss = compute_ss_interaction(perm_vals, window_labels, subgroup_labels)
        if perm_ss >= observed_ss:
            count_ge += 1
    
    p_value = (count_ge + 1) / (n_perm + 1)
    
    # Effect size: partial eta-squared
    ss_total = np.sum((metric_values - grand_mean) ** 2)
    eta_sq = observed_ss / ss_total if ss_total > 0 else 0
    
    return {'p_value': p_value, 'eta_squared': eta_sq, 'ss_interaction': observed_ss}


In [ ]:
# Run H1 test for each metric x protected attribute combination
print('H1 TESTING: Permutation ANOVA (interaction: window x subgroup)')
print(f'Alpha (Bonferroni adjusted): {config.BONFERRONI_ALPHA_H1:.4f}')
print('=' * 70)

h1_results = []

for metric in ['eod', 'dpd', 'ece']:
    for attr in ['race']:  # Focus on race as primary
        subset = fairness_df[
            (fairness_df['protected_attr'] == attr) & 
            (~fairness_df['insufficient_size']) &
            (fairness_df[metric].notna())
        ].copy()
        
        if len(subset) < 10:
            continue
        
        # Aggregate across models for the ANOVA
        agg = subset.groupby(['window', 'group'])[metric].mean().reset_index()
        
        result = permutation_anova_interaction(
            agg[metric].values,
            agg['window'].values,
            agg['group'].values,
            n_perm=config.N_PERMUTATIONS,
            seed=config.PRIMARY_SEED
        )
        
        result['metric'] = metric
        result['attribute'] = attr
        result['significant'] = result['p_value'] < config.BONFERRONI_ALPHA_H1
        h1_results.append(result)
        
        sig = '*' if result['significant'] else ''
        print(f'  {metric.upper():<5} x {attr:<8}: p={result["p_value"]:.4f}{sig}, '
              f'eta^2={result["eta_squared"]:.4f}')

# Check EOD threshold crossing
aa_eod = fairness_df[
    (fairness_df['group'] == 'AfricanAmerican') & 
    (fairness_df['protected_attr'] == 'race')
]
max_eod_by_w4 = aa_eod[aa_eod['window'] <= 4]['eod'].max()
print(f'\nMax AA EOD by W4: {max_eod_by_w4:.4f} (threshold: {config.EOD_VIOLATION_THRESHOLD})')
threshold_crossed = max_eod_by_w4 >= config.EOD_VIOLATION_THRESHOLD
print(f'Threshold crossed: {threshold_crossed}')

# H1 verdict
any_significant = any(r['significant'] for r in h1_results if r['metric'] == 'eod')
h1_supported = any_significant and threshold_crossed
print(f'\nH1 VERDICT: {"SUPPORTED" if h1_supported else "NOT SUPPORTED"}')


## 4. Save Results

In [ ]:
# Save fairness data and H1 results
fairness_df.to_csv('../outputs/metrics/fairness_by_window.csv', index=False)

checkpoint_03 = {
    'fairness_df': fairness_df,
    'h1_results': h1_results,
    'h1_supported': h1_supported,
}
with open('../outputs/checkpoint_03.pkl', 'wb') as f:
    pickle.dump(checkpoint_03, f)
print('Saved: checkpoint_03.pkl + fairness_by_window.csv')
